In [0]:
from pyspark.sql.functions import (
    current_timestamp, lit, col, to_date,
    sum as spark_sum, current_date, to_timestamp
)
from pyspark.sql.window import Window

VOLUME_PATH = "/Volumes/de_workspace26/ecommerce_pawan/raw_files"
CATALOG     = "de_workspace26"
SCHEMA_B    = f"{CATALOG}.bronze_pawan"
SCHEMA_S    = f"{CATALOG}.silver_pawan"
SCHEMA_G    = f"{CATALOG}.gold_pawan"

print("Constants set.")
print("Volume path :", VOLUME_PATH)

In [0]:
display(dbutils.fs.ls(VOLUME_PATH))

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_B}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_S}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_G}")
print("Schemas created:", SCHEMA_B, SCHEMA_S, SCHEMA_G)

In [0]:
orders_raw = spark.read.csv(
    f"{VOLUME_PATH}/orders.csv",
    header=True, inferSchema=True
)
customers_raw = spark.read.csv(
    f"{VOLUME_PATH}/customers.csv",
    header=True, inferSchema=True
)
products_raw = spark.read.csv(
    f"{VOLUME_PATH}/products.csv",
    header=True, inferSchema=True
)

print("orders_raw    rows:", orders_raw.count())     # 205
print("customers_raw rows:", customers_raw.count())  # 20
print("products_raw  rows:", products_raw.count())   # 10

In [0]:
orders_df = (orders_raw
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit("orders.csv"))
)
customers_df = (customers_raw
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit("customers.csv"))
)
products_df = (products_raw
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit("products.csv"))
)

print("Metadata columns added to all three DataFrames.")

In [0]:
orders_df.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{SCHEMA_B}.orders")

customers_df.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{SCHEMA_B}.customers")

products_df.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{SCHEMA_B}.products")

print("Bronze Delta tables written:")
print(f"  {SCHEMA_B}.orders    —", spark.read.table(f"{SCHEMA_B}.orders").count(), "rows")
print(f"  {SCHEMA_B}.customers —", spark.read.table(f"{SCHEMA_B}.customers").count(), "rows")
print(f"  {SCHEMA_B}.products  —", spark.read.table(f"{SCHEMA_B}.products").count(), "rows")

In [0]:
spark.sql(f"""
    ALTER TABLE {SCHEMA_B}.orders
    CHANGE COLUMN order_id SET NOT NULL
""")
print("NOT NULL constraint set on order_id.")

In [0]:
spark.sql(f"DESCRIBE HISTORY {SCHEMA_B}.orders").show(5, truncate=False)


In [0]:
spark.sql(f"DROP TABLE IF EXISTS {SCHEMA_B}.orders_stream")
dbutils.fs.rm(f"{VOLUME_PATH}/_checkpoints/bronze_orders_stream", recurse=True)
dbutils.fs.rm(f"{VOLUME_PATH}/_schema/bronze_orders_stream",      recurse=True)
dbutils.fs.rm(f"{VOLUME_PATH}/orders_raw",                        recurse=True)
print("✅ All previous state cleared.")

In [0]:
ORDERS_RAW_PATH = f"{VOLUME_PATH}/orders_raw"
dbutils.fs.mkdirs(ORDERS_RAW_PATH)

dbutils.fs.cp(f"{VOLUME_PATH}/orders.csv",        f"{ORDERS_RAW_PATH}/orders.csv")
dbutils.fs.cp(f"{VOLUME_PATH}/orders_batch2.csv",  f"{ORDERS_RAW_PATH}/orders_batch2.csv")

files = dbutils.fs.ls(ORDERS_RAW_PATH)
print(f"Files in orders_raw: {len(files)}")
for f in files:
    print(f"  - {f.name}  ({f.size} bytes)")

assert len(files) == 2, f"❌ Expected 2 files but found {len(files)}"
print("✅ Exactly 2 files confirmed.")

In [0]:
df1 = spark.read.csv(f"{ORDERS_RAW_PATH}/orders.csv",        header=True, inferSchema=True)
df2 = spark.read.csv(f"{ORDERS_RAW_PATH}/orders_batch2.csv",  header=True, inferSchema=True)

c1 = df1.count()
c2 = df2.count()
print(f"orders.csv        rows : {c1}")       # 205
print(f"orders_batch2.csv rows : {c2}")       # 30
print(f"Expected total         : {c1 + c2}")  # 235

assert c1 == 205, f"❌ orders.csv expected 205 but got {c1}"
assert c2 == 30,  f"❌ orders_batch2.csv expected 30 but got {c2}"
print("✅ Source row counts verified.")

In [0]:
CHECKPOINT_BRONZE = f"{VOLUME_PATH}/_checkpoints/bronze_orders_stream"
SCHEMA_LOC_BRONZE = f"{VOLUME_PATH}/_schema/bronze_orders_stream"

stream_query = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("cloudFiles.schemaLocation", SCHEMA_LOC_BRONZE)
        .load(ORDERS_RAW_PATH)
    .writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_BRONZE)
        .trigger(availableNow=True)
        .table(f"{SCHEMA_B}.orders_stream")
)
stream_query.awaitTermination()

count = spark.read.table(f"{SCHEMA_B}.orders_stream").count()
print(f"bronze_pawan.orders_stream rows: {count}")
assert count == 235, f"❌ Expected 235 but got {count}"
print("✅ Row count verified: 235")

In [0]:
%sql

select count(*) from bronze_pawan.orders_stream 